# v4 Function Encoder 训练

## Goal

训练 v4 Function Encoder，并同时监控两类互不混用的指标：

1. 从连续先验分布重新采样的随机验证误差；
2. 固定解析问题的真实解重构误差。

v4 中每个样本独立采样
$\sigma_{\theta}\sim U[\sigma_{\theta,\min},\sigma_{\theta,\max}]$ 与
$\sigma_r\sim U[\sigma_{r,\min},\sigma_{r,\max}]$，两者不再固定配对。


## Setup

### Key Assumptions

- Notebook 位于 `polar_annulus_sno_code_v4` 目录中。
- 使用 Miniconda 的 `jax` 环境。
- 固定解析监控采用 $f=0$、$g_n=\cos{\theta}$、$k=1$；它只用于评估，不进入损失函数。
- `RUN_MODE="full"` 会执行正式训练；首次迁移到服务器时可先改为 `"smoke"`。


In [ ]:
from pathlib import Path
import json
import sys

import jax
import matplotlib.pyplot as plt
import numpy as np

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "config_polar.py").exists():
    PROJECT_DIR = Path("/path/to/polar_annulus_sno_code_v4")
if not (PROJECT_DIR / "config_polar.py").exists():
    raise FileNotFoundError("请将 PROJECT_DIR 改为 v4 代码目录。")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from config_polar import PolarAnnulusConfig
from data_polar import sample_sigma_pairs
from train_polar import train_fe

print("Project directory:", PROJECT_DIR)
print("JAX devices:", jax.devices())


## Steps

### 1. Configure the v4 experiment


In [ ]:
RUN_MODE = "full"  # "smoke" or "full"
OUTPUT_ROOT = PROJECT_DIR / "out_polar_annulus_sno_v4"

if RUN_MODE == "smoke":
    cfg = PolarAnnulusConfig(
        sigma_theta_range=(1.0, 2.0),
        sigma_r_range=(1.5, 2.5),
        sample_size=2,
        prior_generation_chunk_size=1,
        hidden_bnn=8,
        theta_size=8,
        radial_size=4,
        random_probe_points=4,
        normalizer_batches=1,
        n_basis=8,
        trunk_width=16,
        trunk_depth=2,
        fe_physics_points=2,
        cnn_channels=(4, 4, 4),
        cnn_kernel_size=(3, 3),
        cnn_stride=(1, 2),
        cnn_dense_width=16,
        transformer_dim=8,
        transformer_heads=2,
        transformer_layers=1,
        transformer_mlp_dim=16,
        seq_chunks=2,
        cond_chunks=2,
        fe_steps=2,
        ol_steps=2,
        fe_log_interval=1,
        fe_checkpoint_interval=2,
        fe_eval_sample_size=2,
        fe_eval_probe_points=4,
        fe_exact_eval_interval=1,
        exact_eval_radial_size=4,
        exact_eval_theta_size=8,
        exact_eval_save_figure=False,
        out_dir=str(OUTPUT_ROOT),
        run_name="polar_v4_smoke",
    )
else:
    cfg = PolarAnnulusConfig(
        r_inner=0.2,
        r_outer=1.0,
        k_min=1.0,
        k_max=1.0,
        sigma_theta_range=(3.0, 7.0),
        sigma_r_range=(3.0, 7.0),
        sample_size=768,
        prior_generation_chunk_size=256,
        hidden_bnn=256,
        theta_size=128,
        radial_size=32,
        random_probe_points=1024,
        normalizer_batches=100,
        n_basis=512,
        trunk_width=512,
        trunk_depth=5,
        fe_phys_weight=0.0,
        fe_physics_points=128,
        cnn_channels=(32, 64, 128),
        cnn_kernel_size=(3, 5),
        cnn_stride=(1, 2),
        cnn_dense_width=1024,
        transformer_dim=512,
        transformer_heads=8,
        transformer_layers=4,
        transformer_mlp_dim=2048,
        seq_chunks=32,
        cond_chunks=32,
        fe_steps=500_000,
        ol_steps=300_000,
        fe_lr=1.0e-3,
        fe_b1=0.5,
        fe_b2=0.9,
        weight_decay=1.0e-6,
        fe_log_interval=500,
        fe_checkpoint_interval=10_000,
        fe_eval_sample_size=16,
        fe_eval_probe_points=256,
        fe_exact_eval_interval=5_000,
        exact_eval_radial_size=64,
        exact_eval_theta_size=128,
        exact_eval_k=1.0,
        exact_eval_mode=1,
        exact_eval_phase=0.0,
        exact_eval_amplitude=1.0,
        exact_eval_save_figure=True,
        seed=0,
        out_dir=str(OUTPUT_ROOT),
        run_name="polar_v4",
    )

print(json.dumps(cfg.__dict__, indent=2, default=list))
print("Total FE batch size:", cfg.effective_batch_size)
print("Output directory:", cfg.output_dir)


### 2. Verify independent scale sampling


In [ ]:
sigma_theta, sigma_r = sample_sigma_pairs(
    jax.random.PRNGKey(2026), 4096, cfg
)
sigma_theta = np.asarray(sigma_theta)
sigma_r = np.asarray(sigma_r)

print("sigma_theta range:", sigma_theta.min(), sigma_theta.max())
print("sigma_r range:", sigma_r.min(), sigma_r.max())
print("sample correlation:", np.corrcoef(sigma_theta, sigma_r)[0, 1])

fig, ax = plt.subplots(figsize=(4.2, 3.6), layout="constrained")
ax.scatter(sigma_theta, sigma_r, s=5, alpha=0.25, edgecolors="none")
ax.set(
    xlabel=r"$\sigma_{	heta}$",
    ylabel=r"$\sigma_r$",
    title="Independent v4 prior-scale samples",
)
ax.grid(alpha=0.2)
plt.show()


### 3. Train FE

训练过程中会写入 `fe_training_history.*`、`fe_exact_history.*`，并按配置保存 `exact_monitor/fe_step_*.png`。


In [ ]:
fe_state, normalizer = train_fe(cfg)


## Checks

### 4. Plot saved random and analytic histories


In [ ]:
def load_history(path):
    if not path.exists():
        print("Not found:", path)
        return None
    return np.atleast_1d(np.genfromtxt(path, delimiter=",", names=True))


random_history = load_history(cfg.output_dir / "fe_training_history.csv")
exact_history = load_history(cfg.output_dir / "fe_exact_history.csv")

fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.4), layout="constrained")
if random_history is not None:
    axes[0].semilogy(
        random_history["step"],
        random_history["eval_p_relative_l2"],
        label="random P",
    )
    axes[0].semilogy(
        random_history["step"],
        random_history["eval_f_relative_l2"],
        label="random f",
    )
if exact_history is not None:
    axes[1].semilogy(
        exact_history["step"],
        exact_history["p_area_relative_l2"],
        label="exact P, area weighted",
    )
    axes[1].semilogy(
        exact_history["step"],
        exact_history["f_zero_rmse"],
        label="exact case f=0 RMSE",
    )
for axis in axes:
    axis.set_xlabel("Step")
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
axes[0].set_title("Fresh random validation")
axes[1].set_title("Fixed analytic monitor")
plt.show()


## Next Steps

确认 `fe_params.msgpack`、归一化文件与两份历史均已生成后，再运行 `train_operator_polar_v4.ipynb`。固定解析监控属于 validation monitor；最终结论仍应使用未参与调参的解析案例和随机测试集。
